# ☀️ LAB 1 · Análisis Exploratorio de Datos (EDA) de producción fotovoltaica

**Ciencia de datos aplicada a sistemas energéticos**. Grado en Ingeniería de la Energía · EEBE – UPC

---

## ¿Qué es el EDA y para qué sirve?

El **Análisis Exploratorio de Datos** (*Exploratory Data Analysis*, EDA) es el primer contacto serio con un conjunto de datos.
Antes de construir cualquier modelo de predicción, necesitamos **entender** los datos:

- ¿Cuántos datos hay y qué significa cada variable?
- ¿Faltan datos? ¿Hay valores raros?
- ¿Cuál es el valor típico de cada variable y cuánto varía?
- ¿Qué forma tiene su distribución?
- ¿Cómo se relacionan las variables entre sí y con el tiempo?

Para responder usaremos dos herramientas que ya habéis visto en la presentación:
**estadística descriptiva** (números que resumen los datos) y **estadística visual** (gráficos).

> 💬 *Un buen EDA cuenta una historia basada en los datos.*

<img src="Figures/eda.jpg" alt="Pasos del EDA" style="width: 650px;"/>

<div style="background-color:#eaf2fc; border-left:6px solid #2a78d6; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**🎯 Objetivo de esta sesión**

Aplicar, paso a paso, las herramientas del EDA a los datos **horarios de una instalación fotovoltaica (FV)**,
junto con datos meteorológicos medidos en el mismo lugar.

Este notebook es una **guía de referencia**.

</div>

## 📑 Índice

| Sección | Contenido | Conceptos de la presentación |
|---|---|---|
| **1. Preparación** | Librerías, estilo de los gráficos y carga de datos | — |
| **2. Conocer los datos** | Tamaño, significado de las variables, datos que faltan, huecos en el tiempo, nuevas variables | — |
| **3. Estadística descriptiva** | Media, mediana, moda, rango, varianza, desviación típica, cuartiles, asimetría y curtosis | Tendencia central · Dispersión · Robustez |
| **4. Gráficos univariantes** | Histograma, densidad, diagrama de caja y serie temporal | Estadística visual (una variable) |
| **5. Gráficos multivariantes** | Dispersión, correlaciones, matriz de dispersión, mapa de calor y cajas agrupadas | Estadística visual (varias variables) |
| **6. Conclusiones** | Resumen de lo aprendido y «receta» del EDA | — |

**Cómo leer este notebook:**
- 🟦 Las cajas **azules** explican *qué vamos a hacer y por qué*.
- 🟩 Las cajas **verdes** interpretan *lo que vemos* en cada resultado o gráfico.
- 🟨 Las cajas **amarillas** destacan *ideas clave* y errores frecuentes.

---
# 1. Preparación

<div style="background-color:#eaf2fc; border-left:6px solid #2a78d6; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**1.1 Importar las librerías**

Solo necesitamos dos librerías:

- **`pandas`** (`pd`): para leer y manipular tablas de datos (los llamados *DataFrames*).
- **`matplotlib.pyplot`** (`plt`): para dibujar gráficos.

</div>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

<div style="background-color:#eaf2fc; border-left:6px solid #2a78d6; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**1.2 Definir el estilo de los gráficos**

Definimos **una sola vez** unos colores y un aspecto común para todos los gráficos.
Así no tenemos que repetir estas opciones cada vez y todos los gráficos quedan coherentes.

</div>

In [ ]:
# Colores que usaremos en todo el notebook
AZUL = '#2a78d6'
NARANJA = '#eb6834'
VERDE = '#1baf7a'
VIOLETA = '#6250d6'
GRIS = '#9a9a9a'

# Aspecto general de los gráficos
plt.rcParams['figure.figsize'] = (10, 4.5)    # tamaño por defecto: (ancho, alto)
plt.rcParams['axes.grid'] = True              # rejilla de fondo...
plt.rcParams['grid.alpha'] = 0.3              # ...pero suave
plt.rcParams['axes.spines.top'] = False       # sin borde superior
plt.rcParams['axes.spines.right'] = False     # sin borde derecho
plt.rcParams['font.size'] = 11

# Mostrar los números de las tablas con 2 decimales
pd.set_option('display.float_format', '{:.2f}'.format)

<div style="background-color:#eaf2fc; border-left:6px solid #2a78d6; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**1.3 Cargar los datos**

Los datos están en el fichero `Data/01_eda_pv.csv`. Es un fichero de texto en el que **cada fila es una hora**
y las columnas están separadas por punto y coma (`;`). Por eso indicamos `sep=';'`.

> Si al cargar un CSV todo aparece en una sola columna, casi siempre es porque el separador no es el correcto.

</div>

In [ ]:
df = pd.read_csv('Data/01_eda_pv.csv', sep=';')

---
# 2. Conocer los datos

Antes de calcular nada, respondemos a preguntas muy básicas: *¿cuántos datos hay?, ¿qué significa cada columna?,
¿falta algo?, ¿están las fechas bien?* Saltarse este paso es la causa más habitual de conclusiones erróneas.

<div style="background-color:#eaf2fc; border-left:6px solid #2a78d6; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**2.1 ¿Cuántos datos hay?**

`df.shape` devuelve el número de **filas** (aquí, horas) y de **columnas** (variables).
Después, `df.head()` muestra las 5 primeras filas para hacernos una idea del contenido.

</div>

In [ ]:
print('Filas (horas):        ', df.shape[0])
print('Columnas (variables): ', df.shape[1])

In [ ]:
df.head()

<div style="background-color:#eaf2fc; border-left:6px solid #2a78d6; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**2.2 ¿Qué significa cada variable?**

Es fundamental conocer **el significado y la unidad** de cada columna. Para que el código sea más fácil de leer,
además **renombramos** las columnas al castellano con `rename`:

| Columna original | Nuevo nombre | Significado | Unidad |
|---|---|---|---|
| `localhour` | `fecha_hora` | Fecha y hora local de la medida | — |
| `pressure` | `presion` | Presión atmosférica | hPa |
| `temperature` | `temperatura` | Temperatura del aire | °C |
| `humidity` | `humedad` | Humedad relativa | % |
| `windspeed` | `viento` | Velocidad del viento | (no indicada en la fuente) |
| `rain` | `lluvia` | Precipitación en la hora | mm |
| `radiation` | `radiacion` | Radiación solar | W/m² |
| `pvgen` | `produccion` | Energía generada por la instalación FV en esa hora | kWh |

</div>

In [ ]:
nombres_nuevos = {
    'localhour': 'fecha_hora',
    'pressure': 'presion',
    'temperature': 'temperatura',
    'humidity': 'humedad',
    'windspeed': 'viento',
    'rain': 'lluvia',
    'radiation': 'radiacion',
    'pvgen': 'produccion',
}
df = df.rename(columns=nombres_nuevos)
df.head()

<div style="background-color:#eaf2fc; border-left:6px solid #2a78d6; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**2.3 ¿Qué tipo de dato tiene cada columna?**

`df.dtypes` indica cómo ha interpretado pandas cada columna:
`float64` = número decimal, `int64` = número entero, `object` (o `str` en las versiones más recientes de pandas) = texto.

La columna `fecha_hora` se ha leído como **texto**. Para poder trabajar con fechas (extraer el mes, la hora,
agrupar por días...) hay que convertirla a tipo **fecha** con `pd.to_datetime`. En el argumento `format`
le explicamos cómo está escrita la fecha: `%d/%m/%Y %H:%M` significa *día/mes/año hora:minuto*.

</div>

In [ ]:
df.dtypes

In [ ]:
df['fecha_hora'] = pd.to_datetime(df['fecha_hora'], format='%d/%m/%Y %H:%M')

# Usamos la fecha como "índice" de la tabla (la etiqueta de cada fila).
# Esto facilita mucho trabajar con series temporales.
df = df.set_index('fecha_hora')

print('Primer dato:', df.index.min())
print('Último dato:', df.index.max())
df.dtypes

<div style="background-color:#e8f6ef; border-left:6px solid #1baf7a; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**🔎 ¿Qué vemos?**

Los datos van del **19 de febrero de 2015** al **1 de mayo de 2018**: algo más de **3 años** de datos horarios.
Ahora la fecha es el índice de la tabla y el resto de columnas son numéricas (`float64`).

</div>

<div style="background-color:#eaf2fc; border-left:6px solid #2a78d6; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**2.4 ¿Faltan valores? ¿Hay filas repetidas?**

Las celdas vacías aparecen en pandas como **`NaN`** (*Not a Number*). Con `df.isna().sum()` contamos cuántas hay en cada columna.
También comprobamos que no haya **horas repetidas** en el índice.

</div>

In [ ]:
print('Horas repetidas:', df.index.duplicated().sum())
print()
print('Valores vacíos en cada columna:')
df.isna().sum()

<div style="background-color:#e8f6ef; border-left:6px solid #1baf7a; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**🔎 ¿Qué vemos?**

- No hay horas repetidas.
- Faltan muy pocos valores sueltos: como mucho 4 (en la presión) de más de 25 000. Es una cantidad muy pequeña.
- La producción no tiene ningún valor vacio.

</div>

<div style="background-color:#eaf2fc; border-left:6px solid #2a78d6; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**2.5 ¿Hay huecos en el tiempo?**

Que no haya celdas vacías **no significa** que no falten datos: puede que falten **filas enteras** (horas en las que no se registró nada).
En datos horarios esto es muy habitual (cortes del sistema de medida, cambios de hora...).

Para comprobarlo, generamos con `pd.date_range` la lista de **todas las horas** que debería haber entre el primer y el último dato,
y la comparamos con las que tenemos.

</div>

In [ ]:
todas_las_horas = pd.date_range(df.index.min(), df.index.max(), freq='h')

print('Horas que debería haber:', len(todas_las_horas))
print('Horas que hay:          ', len(df))
print('Horas que faltan:       ', len(todas_las_horas) - len(df))

<div style="background-color:#eaf2fc; border-left:6px solid #2a78d6; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**Hacemos visibles los huecos**

Con `df.asfreq('h')` obligamos a la tabla a tener **una fila por cada hora**. Las horas que faltaban aparecen ahora como filas con `NaN`.

¿Por qué es útil? Porque así:
- los gráficos temporales mostrarán los huecos (en lugar de unir con una línea puntos muy alejados),
- y operaciones como «el valor de la hora anterior» o «el valor de ayer a la misma hora» serán correctas.

Después contamos cuántas horas faltan en cada mes con `resample('MS')` (MS = *Month Start*, agrupar por meses).

</div>

In [ ]:
df = df.asfreq('h')
print('Filas ahora:', len(df))

# Horas sin dato de producción en cada mes
huecos_por_mes = df['produccion'].isna().resample('MS').sum()

plt.bar(huecos_por_mes.index, huecos_por_mes.values, width=25, color=NARANJA)
plt.title('Horas sin datos en cada mes')
plt.ylabel('Horas sin datos')
plt.show()

# Mostramos solo los meses que tienen algún hueco
huecos_por_mes[huecos_por_mes > 0]

<div style="background-color:#e8f6ef; border-left:6px solid #1baf7a; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**🔎 ¿Qué vemos?**

- Faltan **2 986 horas** (≈ 11 % del periodo), aunque `isna()` no las detectaba: eran **filas que no existían**.
- Casi todas están en **dos grandes huecos** de unos dos meses: de finales de julio a principios de octubre de **2015**
  y de finales de diciembre de **2017** a principios de marzo de **2018**.
- Además hay algunas horas sueltas que faltan (por ejemplo, 1 hora en marzo de 2015, 2016 y 2017).

**¿Por qué importa?** Si no lo supiéramos, podríamos calcular, por ejemplo, la «producción total de 2015» sin el verano y llegar a conclusiones equivocadas.

</div>

In [ ]:
# Miramos de cerca uno de los huecos de 1 hora: la madrugada del 13 de marzo de 2016
df.loc['2016-03-13 00:00':'2016-03-13 04:00']

<div style="background-color:#e8f6ef; border-left:6px solid #1baf7a; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**🔎 ¿Qué vemos?**

Falta justo la hora de las **02:00**: es el **cambio al horario de verano**. Ese día los relojes saltan de las 02:00 a las 03:00,
así que en hora local esa hora **no existe**. No es un fallo del sensor.

Por las fechas (segundo domingo de marzo), esta instalación sigue el calendario de cambio de hora de Estados Unidos.
En España el cambio es el **último domingo de marzo** (y el último de octubre, cuando una hora se repite).
**Tenedlo en cuenta con vuestros datos de consumo.**

</div>

<div style="background-color:#fff5dc; border-left:6px solid #eda100; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**💡 ¿Qué hacemos con los datos que faltan?**

Hay varias opciones y **ninguna es siempre la mejor**:

- **Dejarlos como `NaN`**: pandas los ignora al calcular medias, medianas, etc. Es lo más honesto en un EDA.
- **Eliminar filas** con `df.dropna()`: sencillo, pero se pierde información (y huecos en el tiempo).
- **Rellenar (imputar)**: con el valor anterior (`df.ffill()`), con el siguiente (`df.bfill()`) o interpolando (`df.interpolate()`).
  Razonable para huecos **cortos**; peligroso para huecos de semanas (nos inventaríamos dos meses de datos).

**En este notebook los dejamos como `NaN`**: estamos explorando, no modelando.

</div>

<div style="background-color:#eaf2fc; border-left:6px solid #2a78d6; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**2.6 Crear nuevas variables a partir de la fecha**

La fecha «esconde» información muy valiosa: la hora del día, el mes, la estación del año, si es fin de semana...
Gracias a que el índice es de tipo fecha, podemos extraerla fácilmente con `df.index.hour`, `df.index.month`, etc.

También creamos:
- **`estacion`**: traduciendo cada mes a su estación con un diccionario y `map`.
- **`es_de_dia`**: `True` si hay radiación solar (> 0 W/m²) y `False` si es de noche.

</div>

In [ ]:
df['anyo'] = df.index.year
df['mes'] = df.index.month
df['hora'] = df.index.hour
df['dia_semana'] = df.index.dayofweek           # 0 = lunes, ..., 6 = domingo
df['dia_del_any'] = df.index.dayofyear         # 1 a 366
df['fin_de_semana'] = df['dia_semana'] >= 5     # True si es sábado o domingo

estaciones = {12: 'Invierno', 1: 'Invierno', 2: 'Invierno',
              3: 'Primavera', 4: 'Primavera', 5: 'Primavera',
              6: 'Verano', 7: 'Verano', 8: 'Verano',
              9: 'Otoño', 10: 'Otoño', 11: 'Otoño'}
df['estacion'] = df['mes'].map(estaciones)

df['es_de_dia'] = df['radiacion'] > 0

# Comprobamos el resultado en un amanecer de verano
df.loc['2016-06-21 04:00':'2016-06-21 09:00']

<div style="background-color:#e8f6ef; border-left:6px solid #1baf7a; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**🔎 ¿Qué vemos?**

- De 04:00 a 06:00 la radiación es 0 → `es_de_dia = False`. A partir de las 07:00 hay radiación → `True`.
- El 21/06/2016 fue martes (`dia_semana = 1`), por eso `fin_de_semana = False`, y es junio, así que `estacion = Verano`.
- Fijaos en las 06:00: la instalación ya produce un poquito (0,002 kWh, que la tabla muestra como 0.00 por el redondeo)
  aunque la radiación medida sea 0. La separación día/noche con la radiación no es perfecta al amanecer, pero es suficiente para explorar los datos.

</div>

<div style="background-color:#eaf2fc; border-left:6px solid #2a78d6; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**2.7 Primer resumen estadístico**

`describe()` calcula de golpe los estadísticos más importantes de cada variable numérica:
número de datos (`count`), media (`mean`), desviación típica (`std`), mínimo, cuartiles (25 %, 50 %, 75 %) y máximo.
Con `.T` (de *transponer*) ponemos cada variable en una fila, que se lee mejor.

Guardamos en una lista las variables «físicas» que vamos a analizar, para reutilizarla más adelante.

</div>

In [ ]:
variables = ['presion', 'temperatura', 'humedad', 'viento', 'lluvia', 'radiacion', 'produccion']

df[variables].describe().T

<div style="background-color:#e8f6ef; border-left:6px solid #1baf7a; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**🔎 ¿Qué vemos?**

- **Radiación y producción**: la mediana es casi 0 (3 W/m² y 0,01 kWh) mientras que la media es mucho mayor (178 W/m² y 1,49 kWh).
  Es la primera pista de algo evidente: **la mitad de las horas son de noche**.
- **Lluvia**: Q1, mediana y Q3 valen 0 y el máximo es 32,8 mm. Casi nunca llueve, pero cuando llueve puede ser intenso.
- **Temperatura** entre −6 y 40 °C. **Presión** muy estable (desviación típica de solo 6 hPa sobre unos 1016 hPa).
- **Producción** máxima: 9,54 kWh en una hora, que es aproximadamente la potencia máxima que alcanza la instalación (unos 9,5 kW).
- `count` es menor que el número de filas porque `describe()` **no cuenta los `NaN`** (los huecos).

</div>

---
# 3. Estadística descriptiva

> *La estadística descriptiva se centra en explicar lo que realmente muestran los datos recopilados.*

A partir de aquí nos centramos en la variable más importante: la **producción FV** (`produccion`, en kWh).
Vamos a responder a tres preguntas:

1. **Tendencia central:** ¿cuál es el valor *típico*?
2. **Dispersión:** ¿cuánto varían los datos alrededor de ese valor típico?
3. **Forma:** ¿la distribución es simétrica? ¿tiene colas largas? (**asimetría** y **curtosis**)

## 3.1 Tendencia central: ¿qué es una producción «típica»?

<div style="background-color:#eaf2fc; border-left:6px solid #2a78d6; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**Media, mediana y moda**

- **Media** (`mean`): suma de todos los valores dividida por el número de valores. Es el «punto de equilibrio»: **cada valor tira de ella**, también los extremos.
- **Mediana** (`median`): el valor que deja la **mitad de los datos por debajo** y la otra mitad por encima.
- **Moda** (`mode`): el valor **más frecuente**. En variables continuas (con decimales) casi nunca se repite exactamente el mismo valor,
  así que primero **redondeamos** (aquí a kWh enteros con `round(0)`) y buscamos la «banda» más frecuente, como en la presentación.

> `f'...'` permite mezclar texto y números. `{media:.2f}` significa «escribe `media` con 2 decimales».

</div>

In [ ]:
media = df['produccion'].mean()
mediana = df['produccion'].median()
moda = df['produccion'].round(0).mode()[0]

print(f'Media:   {media:.2f} kWh')
print(f'Mediana: {mediana:.2f} kWh')
print(f'Moda:    {moda:.0f} kWh  (valor redondeado más frecuente)')

<div style="background-color:#e8f6ef; border-left:6px solid #1baf7a; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**🔎 ¿Qué vemos?**

¡Tres valores muy diferentes! Media = 1,49 kWh, mediana = 0,01 kWh y moda = 0 kWh. ¿Cuál es el valor «típico»?

**Ninguno describe bien la realidad**, porque estamos mezclando **dos situaciones muy distintas**: la noche
(producción 0, casi la mitad de las horas) y el día. La mediana y la moda nos dicen «la hora típica no produce nada» y la media
es un valor intermedio que no corresponde ni al día ni a la noche.

👉 Cuando los datos mezclan dos comportamientos, conviene **analizarlos por separado**.

</div>

<div style="background-color:#eaf2fc; border-left:6px solid #2a78d6; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**Repetimos el cálculo solo con las horas de día**

Seleccionamos las filas en las que `es_de_dia` es `True` y guardamos el resultado en una nueva tabla, `de_dia`.
La usaremos a menudo a partir de ahora.

</div>

In [ ]:
de_dia = df[df['es_de_dia']]

print('Horas de día:', len(de_dia))
print(f"Media:   {de_dia['produccion'].mean():.2f} kWh")
print(f"Mediana: {de_dia['produccion'].median():.2f} kWh")
print(f"Moda:    {de_dia['produccion'].round(0).mode()[0]:.0f} kWh")

<div style="background-color:#e8f6ef; border-left:6px solid #1baf7a; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**🔎 ¿Qué vemos?**

- Con las 12 817 horas de día, la **media (2,90 kWh)** y la **mediana (2,43 kWh)** ya son valores representativos.
- La media es mayor que la mediana → hay una cola hacia valores altos (asimetría positiva, lo veremos en el apartado 3.5).
- La **moda sigue siendo 0 kWh** (es decir, entre 0 y 0,5 kWh, por el redondeo): muchas horas «de día» producen muy poco
  (amaneceres, atardeceres y días muy nublados). En variables continuas como esta, la moda **aporta poca información**.

</div>

## 3.2 Dispersión: ¿cuánto varía la producción?

<div style="background-color:#eaf2fc; border-left:6px solid #2a78d6; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**Medidas de dispersión y de posición**

- **Rango** = máximo − mínimo. Fácil, pero depende solo de los dos valores más extremos.
- **Varianza** (`var`): media de las distancias al cuadrado respecto a la media. Sus unidades son kWh², difíciles de interpretar.
- **Desviación típica** (`std`): raíz cuadrada de la varianza. Está en **kWh**, como los datos: es la medida de dispersión más usada.
- **Coeficiente de variación** (CV) $= \dfrac{\text{desviación típica}}{\text{media}} \cdot 100$. Dispersión **relativa** (en %): permite comparar variables con unidades o tamaños diferentes.
- **Cuartiles y percentiles** (`quantile`): el percentil 10 (P10) deja por debajo el 10 % de los datos; Q1 = P25, mediana = P50, Q3 = P75.
- **Rango intercuartílico** (IQR) = Q3 − Q1: la anchura del **50 % central** de los datos. No le afectan los valores extremos.

Lo calculamos para las **horas de día**.

</div>

In [ ]:
x = de_dia['produccion']

rango = x.max() - x.min()
varianza = x.var()
desviacion = x.std()
cv = desviacion / x.mean() * 100

p10 = x.quantile(0.10)
q1 = x.quantile(0.25)
q3 = x.quantile(0.75)
p90 = x.quantile(0.90)
iqr = q3 - q1

print(f'Rango:                    {rango:.2f} kWh  (de {x.min():.2f} a {x.max():.2f})')
print(f'Varianza:                 {varianza:.2f} kWh²')
print(f'Desviación típica:        {desviacion:.2f} kWh')
print(f'Coeficiente de variación: {cv:.0f} %')
print()
print(f'P10 = {p10:.2f} | Q1 = {q1:.2f} | Q3 = {q3:.2f} | P90 = {p90:.2f} kWh')
print(f'Rango intercuartílico (IQR): {iqr:.2f} kWh')

<div style="background-color:#e8f6ef; border-left:6px solid #1baf7a; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**🔎 ¿Qué vemos?**

- La producción de día **varía muchísimo**: la desviación típica (2,39 kWh) es casi tan grande como la media, CV = 82 %.
- El 50 % central de las horas está entre 0,74 y 4,90 kWh (IQR = 4,16 kWh), y el 80 % central entre 0,09 y 6,34 kWh.
- Es lógico: una «hora de día» puede ser un amanecer de invierno o un mediodía despejado de primavera.
- La varianza (5,70 kWh²) es difícil de interpretar por sus unidades. Por eso, en la práctica, se usa la **desviación típica**.

</div>

## 3.3 Dispersión hora a hora

<div style="background-color:#eaf2fc; border-left:6px solid #2a78d6; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**¿Cómo cambian el valor típico y la dispersión a lo largo del día?**

Un único número de dispersión mezcla situaciones muy distintas (mediodía, amanecer, noche...).
Igual que en la presentación, calculamos para **cada hora del día** la mediana y los percentiles P10, P25, P75 y P90
de todos los días disponibles, con `groupby('hora')`.

Luego dibujamos la mediana como una línea y los percentiles como **bandas** con `fill_between`:
- banda oscura (P25–P75): contiene el **50 %** de los días;
- banda clara (P10–P90): contiene el **80 %** de los días.

</div>

In [ ]:
por_hora = df.groupby('hora')['produccion']

p10_h = por_hora.quantile(0.10)
p25_h = por_hora.quantile(0.25)
p50_h = por_hora.quantile(0.50)
p75_h = por_hora.quantile(0.75)
p90_h = por_hora.quantile(0.90)

plt.fill_between(p10_h.index, p10_h, p90_h, color=AZUL, alpha=0.15, label='P10–P90 (80 % de los días)')
plt.fill_between(p25_h.index, p25_h, p75_h, color=AZUL, alpha=0.35, label='P25–P75 (50 % de los días)')
plt.plot(p50_h.index, p50_h, color=AZUL, linewidth=2.5, label='Mediana')

plt.title('Producción FV por hora del día: mediana y bandas de percentiles')
plt.xlabel('Hora del día')
plt.ylabel('Producción (kWh)')
plt.xticks(range(0, 24, 2))
plt.legend(loc='upper left')
plt.show()

<div style="background-color:#e8f6ef; border-left:6px solid #1baf7a; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**🔎 ¿Qué vemos?**

- La producción dibuja una **campana** con el máximo hacia las **13–14 h** (mediana ≈ 5,7 kWh). El mediodía solar cae
  después de las 12 h porque usamos hora local y la mayor parte del año rige el horario de verano.
- Antes de las 6 h y después de las 20 h la producción es 0 y **no hay dispersión**.
- La dispersión es **máxima al mediodía**: a las 13 h, el 80 % de los días produce entre ≈ 1,4 y 7,6 kWh.
- La banda no es simétrica: la mediana está más cerca del P75 que del P25. Los días nublados «estiran» la banda hacia abajo.

👉 Un modelo de predicción debería tener en cuenta que **la incertidumbre depende de la hora del día**.

</div>

## 3.4 Robustez: cuando la mediana es más «honesta» que la media

<div style="background-color:#eaf2fc; border-left:6px solid #2a78d6; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**¿Qué pasa con la media cuando hay valores extremos?**

Tomamos la producción **a las 13:00 de cada día de un mes concreto** y la dibujamos como barras.
Añadimos dos líneas horizontales: la **media** (continua) y la **mediana** (discontinua).

Si unos pocos días son muy distintos al resto (por ejemplo, días muy nublados), veremos cuál de las dos medidas se deja arrastrar más.

</div>

In [ ]:
mes_13h = df[(df['anyo'] == 2016) & (df['mes'] == 5) & (df['hora'] == 13)]['produccion']

plt.bar(mes_13h.index.day, mes_13h.values, color=AZUL)
plt.axhline(mes_13h.mean(), color=NARANJA, linewidth=2.5, label=f'Media = {mes_13h.mean():.2f} kWh')
plt.axhline(mes_13h.median(), color='black', linewidth=2, linestyle='--', label=f'Mediana = {mes_13h.median():.2f} kWh')

plt.title('Producción a las 13:00 en cada día de mayo de 2016')
plt.xlabel('Día del mes')
plt.ylabel('Producción (kWh)')
plt.legend(loc='lower left')
plt.show()

<div style="background-color:#e8f6ef; border-left:6px solid #1baf7a; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**🔎 ¿Qué vemos?**

- La mayoría de los días de mayo de 2016 producen entre 5 y 8,5 kWh a las 13:00.
- Pero hay **tres días muy nublados** (8, 14 y 25 de mayo) con menos de 1 kWh.
- Esos pocos días **arrastran la media hacia abajo** (5,48 kWh), mientras que la **mediana** apenas se mueve (5,95 kWh).

Aquí la mediana describe mejor «un día típico de mayo a las 13:00». Como la media es menor que la mediana, la cola está a la izquierda (asimetría negativa).

</div>

<div style="background-color:#fff5dc; border-left:6px solid #eda100; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**💡 Regla práctica**

**Indica siempre la media y la mediana.** Si son muy diferentes, los datos tienen valores atípicos,
colas largas o mezclan más de un comportamiento (como el día y la noche). La mediana es **robusta**: le afectan poco los extremos.

</div>

---
# 4. Gráficos univariantes

Los números resumen, pero **esconden detalles**. Un gráfico permite ver de un vistazo dónde se concentran los datos,
cuánto se dispersan, qué forma tienen y si hay valores raros. Empezamos con gráficos de **una sola variable**.

## 4.1 Histograma

<div style="background-color:#eaf2fc; border-left:6px solid #2a78d6; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**¿Qué es un histograma?**

Divide el eje horizontal en **intervalos** (*bins*) y dibuja una barra con **cuántos datos** caen en cada intervalo.
Es la **primera mirada** que conviene dar a cualquier variable.

Empezamos con un histograma de **todas las variables** a la vez con `df.hist()`.

</div>

In [ ]:
df[variables].hist(bins=50, figsize=(14, 8), color=AZUL, edgecolor='white')
plt.tight_layout()
plt.show()

<div style="background-color:#e8f6ef; border-left:6px solid #1baf7a; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**🔎 ¿Qué vemos?**

- **Presión**: forma de campana, casi simétrica.
- **Temperatura** y **humedad**: cola hacia la izquierda (asimetría negativa).
- **Viento**: barras separadas porque solo toma **valores enteros** (0, 1, 2...): se ha medido con poca resolución.
- **Lluvia, radiación y producción**: un gran pico en 0 (horas sin lluvia o de noche) que domina todo el gráfico.
- Los «dientes» (barras vacías) en temperatura y humedad **no son reales**: estas variables se registran con números enteros
  y algunos intervalos del histograma no contienen ningún entero. Es un efecto del número de intervalos (lo vemos más abajo).

</div>

<div style="background-color:#eaf2fc; border-left:6px solid #2a78d6; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**Histograma de la producción: noche frente a día**

Separamos las horas de **noche** y de **día** y las dibujamos apiladas en el mismo histograma (`stacked=True`).
Así vemos de dónde sale cada parte de la distribución. Usamos `dropna()` para quitar los vacíos, que `plt.hist` no acepta.

</div>

In [ ]:
noche = df[df['es_de_dia'] == False]['produccion'].dropna()
dia = df[df['es_de_dia'] == True]['produccion'].dropna()

plt.hist([noche, dia], bins=40, stacked=True, color=[GRIS, NARANJA], edgecolor='white',
         label=['Noche (sin radiación)', 'Día (con radiación)'])
plt.title('Producción FV horaria: noche frente a día')
plt.xlabel('Producción (kWh)')
plt.ylabel('Número de horas')
plt.legend()
plt.show()

<div style="background-color:#e8f6ef; border-left:6px solid #1baf7a; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**🔎 ¿Qué vemos?**

- La barra gigante de 0 kWh está formada casi por completo por **horas de noche** (gris).
- La parte naranja de esa misma barra son horas de día con producción casi nula: amaneceres, atardeceres y días muy nublados.
- Todo lo demás (de 0,25 a 9,5 kWh) son horas de día.

Igual que en la presentación (demanda de noche frente al resto del día): **mismos datos, dos regímenes**.

</div>

<div style="background-color:#eaf2fc; border-left:6px solid #2a78d6; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**Histograma de las horas de día, con media y mediana**

Nos quedamos solo con las horas de día y marcamos la media y la mediana con líneas verticales (`axvline`).

</div>

In [ ]:
plt.hist(dia, bins=40, color=NARANJA, edgecolor='white')
plt.axvline(dia.mean(), color='black', linewidth=2, label=f'Media = {dia.mean():.2f} kWh')
plt.axvline(dia.median(), color='black', linewidth=2, linestyle='--', label=f'Mediana = {dia.median():.2f} kWh')

plt.title('Producción FV en las horas de día')
plt.xlabel('Producción (kWh)')
plt.ylabel('Número de horas')
plt.legend()
plt.show()

<div style="background-color:#e8f6ef; border-left:6px solid #1baf7a; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**🔎 ¿Qué vemos?**

- Incluso sin la noche, la distribución **no es una campana**: hay muchas horas con poca producción (amanecer, atardecer, nubes),
  una «meseta» entre 1 y 6,5 kWh y una caída rápida a partir de unos 7 kWh, cuando la instalación se acerca a su máximo (≈ 9,5 kWh).
- Media (2,90) > mediana (2,43): **asimetría positiva**.

</div>

<div style="background-color:#fff5dc; border-left:6px solid #eda100; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**💡 Cuidado con el número de intervalos (*bins*)**

El aspecto de un histograma depende del número de intervalos. Con **pocos** se pierden detalles; con **demasiados** aparece «ruido».
Prueba siempre un par de valores antes de sacar conclusiones.

</div>

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(14, 3.5), sharey=False)

numero_bins = [5, 30, 200]
for i in range(3):
    ejes[i].hist(dia, bins=numero_bins[i], color=NARANJA, edgecolor='white')
    ejes[i].set_title(f'{numero_bins[i]} intervalos')
    ejes[i].set_xlabel('Producción (kWh)')

plt.tight_layout()
plt.show()

<div style="background-color:#e8f6ef; border-left:6px solid #1baf7a; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**🔎 ¿Qué vemos?**

- Con **5 intervalos** se pierde la forma: no se ve el pico cerca de 0 ni la caída a partir de 7 kWh.
- Con **30 intervalos** la forma se ve clara.
- Con **200 intervalos** las barras son tan finas que aparece «ruido» y cuesta leer el gráfico.

Para miles de datos, entre 20 y 50 intervalos suele ser un buen punto de partida.

</div>

## 4.2 Gráfico de densidad

<div style="background-color:#eaf2fc; border-left:6px solid #2a78d6; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**¿Qué es un gráfico de densidad?**

Es como un **histograma suavizado**: en lugar de barras, una curva continua. Su ventaja es que no depende del número de intervalos
y permite **superponer varios grupos** en un mismo gráfico sin que se tapen. El área bajo cada curva vale 1,
así que comparamos **formas**, no cantidades.

Dibujamos la densidad de la producción de día y de la temperatura **para cada estación del año**, con `plot(kind='kde')`.
Para recorrer las estaciones y sus colores a la vez usamos un bucle `for` con `zip`.

> `kind='kde'` usa internamente la librería `scipy`, que viene instalada con Anaconda.

</div>

In [ ]:
orden_estaciones = ['Invierno', 'Primavera', 'Verano', 'Otoño']
colores_estaciones = [AZUL, VERDE, NARANJA, VIOLETA]

fig, ejes = plt.subplots(1, 2, figsize=(14, 4.5))

for estacion, color in zip(orden_estaciones, colores_estaciones):
    datos = de_dia[de_dia['estacion'] == estacion]
    datos['produccion'].plot(kind='kde', ax=ejes[0], color=color, linewidth=2, label=estacion)
    datos['temperatura'].plot(kind='kde', ax=ejes[1], color=color, linewidth=2, label=estacion)

ejes[0].set_title('Producción FV (horas de día)')
ejes[0].set_xlabel('Producción (kWh)')
ejes[0].set_xlim(-1, 11)
ejes[1].set_title('Temperatura (horas de día)')
ejes[1].set_xlabel('Temperatura (°C)')
ejes[1].set_xlim(-10, 45)
for eje in ejes:
    eje.set_ylabel('Densidad')
    eje.legend()

plt.tight_layout()
plt.show()

<div style="background-color:#e8f6ef; border-left:6px solid #1baf7a; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**🔎 ¿Qué vemos?**

**Producción**
- En todas las estaciones hay un pico cerca de 0–1 kWh (horas con poca luz).
- El **verano** tiene más horas en torno a 5–6 kWh (días largos y despejados)...
- ...pero, curiosamente, su cola derecha es la **más corta**: en verano solo ≈ 1 % de las horas de día superan los 7 kWh,
  frente a ≈ 8 % en invierno. La explicación es física: **los paneles FV rinden menos cuando hace calor**
  (la eficiencia de las células baja al subir su temperatura). Lo confirmaremos en el diagrama de dispersión.

**Temperatura**
- Las curvas están claramente separadas: el verano se concentra entre 25 y 37 °C; el invierno es mucho más ancho (de 0 a 25 °C)
  y tiene **dos máximos** (días fríos y días templados). Primavera y otoño se solapan en gran parte.

</div>

<div style="background-color:#fff5dc; border-left:6px solid #eda100; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**💡 Un detalle de las curvas de densidad**

Al suavizar, la curva puede «salirse» por debajo de 0 kWh aunque **no existan** producciones negativas.
Es un efecto del método de dibujo, no de los datos. Por eso conviene mirar también el histograma.

</div>

## 4.3 Diagrama de caja (*boxplot*) y valores atípicos

<div style="background-color:#eaf2fc; border-left:6px solid #2a78d6; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**¿Cómo se lee un diagrama de caja?**

Resume una distribución con **cinco números** y marca los posibles **valores atípicos**:

- **Caja**: va del primer cuartil (Q1) al tercero (Q3). Contiene el **50 % central** de los datos (su altura es el IQR).
- **Línea dentro de la caja**: la **mediana**.
- **Bigotes**: llegan hasta el dato más extremo que esté a menos de **1,5 × IQR** de la caja.
- **Puntos**: datos más allá de los bigotes → **posibles valores atípicos** (*outliers*).

<img src="Figures/boxplot_explained.png" alt="Partes de un diagrama de caja" style="width: 600px;"/>

Primero, un diagrama de caja de **cada variable**, con `plot(kind='box', subplots=True)`.

</div>

In [ ]:
df[variables].plot(kind='box', subplots=True, layout=(2, 4), figsize=(14, 7), sharey=False)
plt.tight_layout()
plt.show()

<div style="background-color:#e8f6ef; border-left:6px solid #1baf7a; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**🔎 ¿Qué vemos?**

- En lluvia, radiación y producción la caja está «aplastada» en 0: la mezcla noche/día deforma el gráfico.
- Aparecen muchos posibles atípicos: presión (por ambos lados), temperatura (por debajo de 2 °C), viento (6 o más),
  lluvia (¡cualquier valor mayor que 0!), radiación y producción (por encima de unos 6,4 kWh).
- **La mayoría no son errores**: el diagrama de caja marca como atípico lo que es poco frecuente para *esa* distribución.

</div>

<div style="background-color:#eaf2fc; border-left:6px solid #2a78d6; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**Diagramas de caja por estación**

Los diagramas de caja son especialmente útiles para **comparar grupos**. Dibujamos la producción de las horas de día en cada estación.
Con un bucle preparamos una **lista** con los datos de cada estación y se la pasamos a `plt.boxplot`.

</div>

In [ ]:
datos_por_estacion = []
for estacion in orden_estaciones:
    datos_por_estacion.append(de_dia[de_dia['estacion'] == estacion]['produccion'].dropna())

plt.boxplot(datos_por_estacion, patch_artist=True,
            boxprops={'facecolor': AZUL, 'alpha': 0.6},
            medianprops={'color': 'black', 'linewidth': 2})
plt.xticks([1, 2, 3, 4], orden_estaciones)
plt.title('Producción FV en horas de día, por estación')
plt.ylabel('Producción (kWh)')
plt.show()

<div style="background-color:#e8f6ef; border-left:6px solid #1baf7a; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**🔎 ¿Qué vemos?**

- La **mediana** sube de invierno (≈ 1,8 kWh) a verano (≈ 3,0 kWh): en verano hay más horas de buena producción.
- Pero el **máximo** (bigote superior) es más bajo en verano (≈ 8,2 kWh) que en invierno y primavera (≈ 9,5 kWh):
  otra vez el efecto de la temperatura sobre los paneles.
- La altura de las cajas (IQR) es parecida en todas las estaciones: la variabilidad hora a hora es grande todo el año.

</div>

<div style="background-color:#eaf2fc; border-left:6px solid #2a78d6; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**Detectar valores atípicos: método del rango intercuartílico (IQR)**

Es la misma regla que usa el diagrama de caja. Un valor es **atípico** si está fuera del intervalo:

$$[\,Q_1 - 1{,}5 \cdot IQR\;,\; Q_3 + 1{,}5 \cdot IQR\,]$$

Lo aplicamos a la **temperatura**, que en el diagrama de caja mostraba puntos por debajo del bigote inferior.

</div>

In [ ]:
t = df['temperatura']

q1_t = t.quantile(0.25)
q3_t = t.quantile(0.75)
iqr_t = q3_t - q1_t
limite_inferior = q1_t - 1.5 * iqr_t
limite_superior = q3_t + 1.5 * iqr_t

atipicos_iqr = df[(t < limite_inferior) | (t > limite_superior)]

print(f'Límites: de {limite_inferior:.1f} °C a {limite_superior:.1f} °C')
print('Horas atípicas (IQR):', len(atipicos_iqr))

<div style="background-color:#eaf2fc; border-left:6px solid #2a78d6; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**¿Cuándo ocurren esos valores atípicos?**

Un valor atípico no es necesariamente un error. Antes de decidir qué hacer, **miramos cuándo ocurren**:
contamos cuántos hay en cada mes con `value_counts()`.

</div>

In [ ]:
atipicos_por_mes = atipicos_iqr['mes'].value_counts().sort_index()

plt.bar(atipicos_por_mes.index, atipicos_por_mes.values, color=AZUL)
plt.title('Horas con temperatura atípica (método IQR), por mes')
plt.xlabel('Mes')
plt.ylabel('Número de horas')
plt.xticks(range(1, 13))
plt.show()

print(f"Temperatura media de esas horas: {atipicos_iqr['temperatura'].mean():.1f} °C")

<div style="background-color:#e8f6ef; border-left:6px solid #1baf7a; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**🔎 ¿Qué vemos?**

- Hay **166 horas** con temperatura atípica (por debajo de 2 °C), todas **entre noviembre y marzo**, sobre todo en diciembre, enero y febrero.
- Su temperatura media es de **−0,8 °C**: son **episodios de frío reales**, poco frecuentes en este lugar, y no errores de medida.

</div>

<div style="background-color:#eaf2fc; border-left:6px solid #2a78d6; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**Otro método: la puntuación Z (*Z-score*)**

La puntuación Z indica **a cuántas desviaciones típicas** de la media está cada valor:

$$Z = \frac{x - \text{media}}{\text{desviación típica}}$$

Se suele considerar atípico un valor con $|Z| > 3$. Este método funciona bien si la distribución se parece a una normal,
porque usa la media y la desviación típica (que **no** son robustas).

</div>

In [ ]:
z = (t - t.mean()) / t.std()
atipicos_z = df[(z > 3) | (z < -3)]

print('Horas atípicas (Z-score):', len(atipicos_z))
print('Horas atípicas (IQR):    ', len(atipicos_iqr))

<div style="background-color:#e8f6ef; border-left:6px solid #1baf7a; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**🔎 ¿Qué vemos?**

- La puntuación Z detecta **45 horas** (temperaturas de −2 °C o menos, en diciembre y enero), muchas menos que el método IQR (166).
- Los dos métodos usan criterios distintos, así que no tienen por qué coincidir. **Ninguno es «el correcto»**:
  lo importante es revisar qué valores marca cada uno y **entender por qué**.

</div>

<div style="background-color:#eaf2fc; border-left:6px solid #2a78d6; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**Cuando el método IQR no sirve: la lluvia**

Veamos qué pasaría si aplicamos el método IQR a la **lluvia**.

</div>

In [ ]:
lluvia = df['lluvia']
iqr_lluvia = lluvia.quantile(0.75) - lluvia.quantile(0.25)

print('Q1 de la lluvia:', lluvia.quantile(0.25), 'mm')
print('Q3 de la lluvia:', lluvia.quantile(0.75), 'mm')
print('IQR de la lluvia:', iqr_lluvia, 'mm')
print('Horas con lluvia > 0 mm (todas serían "atípicas"):', (lluvia > 0).sum())

<div style="background-color:#e8f6ef; border-left:6px solid #1baf7a; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**🔎 ¿Qué vemos?**

Como Q1 = Q3 = 0, el IQR vale 0 y los límites son [0, 0]: **cualquier hora con lluvia (668 horas) sería «atípica»**.

El método IQR **no sirve** para variables en las que la mayoría de los valores son iguales (muy asimétricas).
Por eso nunca hay que aplicar estos métodos a ciegas: siempre hay que **mirar los gráficos y pensar**.

</div>

<div style="background-color:#fff5dc; border-left:6px solid #eda100; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**💡 ¿Eliminamos los valores atípicos?**

**No de forma automática.** Primero hay que preguntarse *por qué* existen:

- Si son **errores** (un sensor averiado, una temperatura de 200 °C, una producción negativa...) → se corrigen o se eliminan.
- Si son **fenómenos reales** (una ola de frío, una tormenta, un apagón...) → se mantienen: son parte de la realidad que queremos entender.

En nuestros datos, las temperaturas atípicas son días reales de frío: **los conservamos**.

</div>

## 4.4 Gráfico de serie temporal

<div style="background-color:#eaf2fc; border-left:6px solid #2a78d6; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**Los datos en orden temporal**

Con datos que dependen del tiempo, la serie temporal es **el primer gráfico que hay que mirar**: muestra tendencias,
ciclos (diarios, semanales, anuales), eventos puntuales y huecos.

Con 28.000 horas el gráfico sería ilegible, así que calculamos la **energía diaria** sumando las 24 horas de cada día con `resample('D').sum()`.
- `min_count=20` exige al menos 20 horas con dato; si no, el día queda vacío (`NaN`). Sin esto, ¡un día sin datos sumaría 0 kWh y parecería un día sin sol!
- Añadimos una **media móvil de 30 días** (`rolling(30).mean()`) para ver mejor la tendencia estacional.

</div>

In [ ]:
energia_diaria = df['produccion'].resample('D').sum(min_count=20)
media_movil = energia_diaria.rolling(30, center=True).mean()

plt.figure(figsize=(14, 4.5))
plt.plot(energia_diaria.index, energia_diaria, color=NARANJA, linewidth=0.8, alpha=0.6, label='Energía diaria')
plt.plot(media_movil.index, media_movil, color='black', linewidth=2, label='Media móvil de 30 días')
plt.title('Energía FV producida cada día')
plt.ylabel('Energía (kWh/día)')
plt.legend(loc='upper left')
plt.show()

<div style="background-color:#e8f6ef; border-left:6px solid #1baf7a; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**🔎 ¿Qué vemos?**

- **Ciclo anual** claro: mínimos en invierno (≈ 20 kWh/día en la media móvil) y máximos en primavera y verano (40–50 kWh/día).
- **Enorme variabilidad de un día a otro**: días de 60–70 kWh junto a días de menos de 10 kWh (nublados o lluviosos; hay 89 días así).
- Los **huecos** de agosto–septiembre de 2015 y enero–febrero de 2018 se ven perfectamente (los encontramos en el apartado 2.5).
- **2016 parece un año mejor** que 2015 y 2017: por ejemplo, en marzo y abril de 2016 la media fue de ≈ 47 kWh/día, frente a 31–39 kWh/día
  en los otros años. ¿Más sol? ¿Paneles más limpios? ¿Cambios en la instalación?

👉 El EDA no lo responde todo, pero **genera buenas preguntas**.

</div>

<div style="background-color:#eaf2fc; border-left:6px solid #2a78d6; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**Zoom: una semana de verano y una de invierno**

Para ver el **ciclo diario**, dibujamos los datos horarios de una semana. Seleccionamos un intervalo de fechas con `df.loc['inicio':'fin']`.
Con `sharey=True` las dos gráficas comparten el eje vertical y se pueden comparar directamente.

</div>

In [ ]:
semana_verano = df.loc['2016-07-04':'2016-07-10', 'produccion']
semana_invierno = df.loc['2016-12-05':'2016-12-11', 'produccion']

fig, ejes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
ejes[0].plot(semana_verano.index, semana_verano, color=NARANJA, linewidth=2)
ejes[0].set_title('Semana del 4 al 10 de julio de 2016')
ejes[1].plot(semana_invierno.index, semana_invierno, color=AZUL, linewidth=2)
ejes[1].set_title('Semana del 5 al 11 de diciembre de 2016')
ejes[0].set_ylabel('Producción (kWh)')
for eje in ejes:
    eje.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

<div style="background-color:#e8f6ef; border-left:6px solid #1baf7a; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**🔎 ¿Qué vemos?**

- **Julio**: siete «campanas» casi idénticas, con picos de 6–7 kWh y producción desde las ≈ 7 h hasta las ≈ 20 h.
- **Diciembre**: días más cortos (≈ 8 h a 17 h) y picos mucho más bajos e irregulares (de 1 a 4,5 kWh), por el sol bajo y las nubes.

El **ciclo diario** es el patrón dominante de los datos: la hora del día será una variable clave para predecir la producción.

</div>

<div style="background-color:#eaf2fc; border-left:6px solid #2a78d6; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**Energía media diaria de cada mes**

Para resumir el **ciclo anual** agrupamos la energía diaria por mes del año (juntando todos los años) y calculamos la media.
Usamos la media de los días y no la suma del mes, porque algunos meses tienen huecos y su suma saldría artificialmente baja.

</div>

In [ ]:
energia_por_mes = energia_diaria.groupby(energia_diaria.index.month).mean()

plt.bar(energia_por_mes.index, energia_por_mes.values, color=NARANJA)
plt.title('Energía FV media diaria según el mes del año')
plt.xlabel('Mes')
plt.ylabel('Energía media (kWh/día)')
plt.xticks(range(1, 13), ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic'])
plt.show()

<div style="background-color:#e8f6ef; border-left:6px solid #1baf7a; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**🔎 ¿Qué vemos?**

- Máximo en **julio** (≈ 46 kWh/día) y **junio** (≈ 43 kWh/día); mínimo en **noviembre** y **diciembre** (≈ 24 y 21 kWh/día).
- De febrero a octubre la producción es bastante estable (35–40 kWh/día). Mayo es algo más bajo que abril (¿más nubes?).
- ⚠️ Cada barra combina varios años y algunos meses (agosto, septiembre, enero, febrero) tienen menos años por culpa de los huecos.

</div>

---
# 5. Gráficos multivariantes

Hasta ahora hemos mirado cada variable por separado. Ahora estudiamos **cómo se relacionan** entre sí.
Esto es clave para decidir, por ejemplo, qué variables usar en un modelo para **predecir** la producción FV.

## 5.1 Diagrama de dispersión (*scatter plot*)

<div style="background-color:#eaf2fc; border-left:6px solid #2a78d6; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**Radiación frente a producción**

Cada punto es una hora: en el eje x la radiación y en el eje y la producción. Así vemos la **forma** de la relación
(lineal, curva...), su **intensidad**, si hay **grupos** y si hay **puntos raros**.

Con el argumento `c=` coloreamos cada punto según una tercera variable (la temperatura). `s` es el tamaño de los puntos
y `alpha` su transparencia (útil cuando hay muchos puntos superpuestos).

</div>

In [ ]:
plt.figure(figsize=(9, 6))
plt.scatter(df['radiacion'], df['produccion'], c=df['temperatura'], cmap='coolwarm', s=5, alpha=0.5)
plt.colorbar(label='Temperatura (°C)')

plt.title('Radiación solar frente a producción FV (cada punto = 1 hora)')
plt.xlabel('Radiación (W/m²)')
plt.ylabel('Producción (kWh)')
plt.show()

<div style="background-color:#e8f6ef; border-left:6px solid #1baf7a; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**🔎 ¿Qué vemos?**

- Relación **casi lineal y muy fuerte**: a más radiación, más producción.
- Aun así hay bastante dispersión: con 600 W/m² la producción va de 2 a 8 kWh.
- **El color revela un patrón**: para la misma radiación, los puntos **azules (frío) quedan arriba** y los **rojos (calor) abajo**.
  Confirma que los paneles rinden menos con calor.
- **Puntos imposibles**: hay una columna vertical en radiación = 0 con producciones de hasta 6 kWh. Son 54 horas, la mayoría de **junio de 2015**
  (por ejemplo, todo el día 22/06/2015): el **sensor de radiación marcó 0** mientras la instalación producía con normalidad.
  Es un **error de medida** que habría que corregir antes de usar estos datos en un modelo.

</div>

<div style="background-color:#eaf2fc; border-left:6px solid #2a78d6; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**Humedad y temperatura frente a producción**

Repetimos con otras dos variables meteorológicas, solo con las horas de día.

</div>

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)

ejes[0].scatter(de_dia['humedad'], de_dia['produccion'], color=AZUL, s=5, alpha=0.2)
ejes[0].set_xlabel('Humedad relativa (%)')
ejes[0].set_ylabel('Producción (kWh)')
ejes[0].set_title('Humedad frente a producción (de día)')

ejes[1].scatter(de_dia['temperatura'], de_dia['produccion'], color=NARANJA, s=5, alpha=0.2)
ejes[1].set_xlabel('Temperatura (°C)')
ejes[1].set_title('Temperatura frente a producción (de día)')

plt.tight_layout()
plt.show()

<div style="background-color:#e8f6ef; border-left:6px solid #1baf7a; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**🔎 ¿Qué vemos?**

- **Humedad**: relación negativa. Con humedad muy alta (> 85 %) casi nunca se superan 2–4 kWh (nubes, niebla, lluvia, primeras horas del día).
  Con humedad baja puede ocurrir de todo.
- **Temperatura**: relación débil y no lineal. Las producciones más altas aparecen entre 15 y 30 °C; por encima de 35 °C el máximo baja.
  Las «columnas» verticales se deben a que la temperatura se registra en números enteros.
- Ojo: la humedad o la temperatura no «causan» directamente la producción; ambas están ligadas a la hora del día y a la nubosidad.

</div>

## 5.2 Mapa de calor de correlaciones

<div style="background-color:#eaf2fc; border-left:6px solid #2a78d6; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**¿Qué es el coeficiente de correlación?**

Es un número entre **−1 y +1** que mide la relación entre dos variables:

- **+1**: relación positiva perfecta (cuando una sube, la otra también).
- **0**: sin relación **lineal**.
- **−1**: relación negativa perfecta (cuando una sube, la otra baja).

Hay dos tipos habituales:
- **Pearson** (por defecto en pandas): mide relaciones **lineales** (en línea recta).
- **Spearman**: mide relaciones **monótonas** (que siempre suben o siempre bajan, aunque sea en curva). Es más robusto frente a atípicos.

<img src="Figures/correlation.png" alt="Pearson frente a Spearman" style="width: 650px;"/>

> ⚠️ **Correlación no implica causalidad**: que dos variables se muevan juntas no significa que una cause la otra.

</div>

<div style="background-color:#eaf2fc; border-left:6px solid #2a78d6; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**Añadimos variables «retardadas»**

En predicción energética, a menudo la mejor pista sobre el futuro es **el pasado de la propia variable**.
Creamos dos variables con `shift`, que desplaza la serie hacia abajo:

- `produccion_hora_anterior`: la producción de **1 hora antes** (`shift(1)`).
- `produccion_ayer`: la producción a **la misma hora del día anterior** (`shift(24)`).

Esto es correcto porque en el apartado 2.5 hicimos que hubiera **una fila por hora** con `asfreq('h')`.

</div>

In [ ]:
df['produccion_hora_anterior'] = df['produccion'].shift(1)
df['produccion_ayer'] = df['produccion'].shift(24)

columnas_corr = ['produccion', 'produccion_hora_anterior', 'produccion_ayer',
                 'radiacion', 'temperatura', 'humedad', 'viento', 'lluvia', 'presion',
                 'hora', 'mes', 'dia_semana']

corr = df[columnas_corr].corr()     # Pearson por defecto
corr['produccion'].sort_values(ascending=False)

<div style="background-color:#eaf2fc; border-left:6px solid #2a78d6; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**Dibujamos la matriz como un mapa de calor**

Con `plt.imshow` cada celda se pinta con un color según su valor: **rojo** = correlación positiva, **azul** = negativa,
**blanco** = sin correlación (con `vmin=-1` y `vmax=1` fijamos la escala de −1 a +1). Con dos bucles `for` escribimos el valor dentro de cada celda.

</div>

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
imagen = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)

ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=90)
ax.set_yticks(range(len(corr.columns)))
ax.set_yticklabels(corr.columns)
ax.grid(False)

for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        valor = corr.iloc[i, j]
        color_texto = 'white' if abs(valor) > 0.6 else 'black'   # texto blanco sobre celdas oscuras
        ax.text(j, i, f'{valor:.2f}', ha='center', va='center', fontsize=8, color=color_texto)

fig.colorbar(imagen, label='Correlación de Pearson')
plt.title('Matriz de correlación (Pearson)')
plt.show()

<div style="background-color:#e8f6ef; border-left:6px solid #1baf7a; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**🔎 ¿Qué vemos?**

- La producción tiene una correlación **muy alta** con la radiación (0,94), con la **producción de la hora anterior** (0,92)
  y con la de **ayer a la misma hora** (0,86). Como en la presentación: **dominan los retardos**.
- **Humedad**: correlación negativa moderada (−0,49).
- **Temperatura** y **viento**: positiva débil (0,35 y 0,32), probablemente porque ambos son mayores en las horas de sol.
- **Presión, lluvia, mes y día de la semana**: ≈ 0.
- Entre las variables de entrada también hay correlaciones fuertes, por ejemplo radiación y producción de la hora anterior (0,92):
  contienen casi la misma información (**multicolinealidad**). Es útil saberlo para elegir variables en un modelo.

</div>

<div style="background-color:#eaf2fc; border-left:6px solid #2a78d6; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**Pearson frente a Spearman**

Calculamos también la correlación de **Spearman** y comparamos, para la producción, ambos coeficientes en una tabla.

</div>

In [ ]:
corr_spearman = df[columnas_corr].corr(method='spearman')

comparacion = pd.DataFrame({
    'Pearson': corr['produccion'],
    'Spearman': corr_spearman['produccion'],
})
comparacion.sort_values('Pearson', ascending=False)

<div style="background-color:#e8f6ef; border-left:6px solid #1baf7a; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**🔎 ¿Qué vemos?**

- Los dos coeficientes son muy parecidos para casi todas las variables.
- Spearman es mayor para la producción de ayer (0,95 frente a 0,86) y la radiación (0,96 frente a 0,94): la relación es
  **monótona** (siempre crece), pero no exactamente **lineal**.

</div>

<div style="background-color:#fff5dc; border-left:6px solid #eda100; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**💡 Correlación ≈ 0 no significa «no hay relación»**

La correlación con la **hora del día** es débil (0,18), y sin embargo en el apartado 3.3 vimos que la hora es **decisiva**:
la producción sube por la mañana y baja por la tarde (forma de campana). Pearson y Spearman solo detectan relaciones
que van **siempre en el mismo sentido**. Por eso, después del mapa de calor, **hay que dibujar siempre las parejas de variables**.

</div>

## 5.3 Matriz de dispersión (*pair plot*)

<div style="background-color:#eaf2fc; border-left:6px solid #2a78d6; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**Todas las parejas de variables de un vistazo**

La **matriz de dispersión** dibuja un diagrama de dispersión para **cada pareja** de variables y, en la diagonal, el histograma de cada una.

Con datos horarios tendríamos demasiados puntos, así que trabajamos con **valores diarios**:
energía total del día, producción máxima horaria, radiación media, temperatura media, humedad media y lluvia total.
Al final eliminamos los días incompletos con `dropna()`.

</div>

In [ ]:
diario = pd.DataFrame()
diario['energia'] = energia_diaria
diario['produccion_max'] = df['produccion'].resample('D').max()
diario['radiacion_media'] = df['radiacion'].resample('D').mean()
diario['temperatura_media'] = df['temperatura'].resample('D').mean()
diario['humedad_media'] = df['humedad'].resample('D').mean()
diario['lluvia_total'] = df['lluvia'].resample('D').sum(min_count=20)
diario = diario.dropna()

print('Días completos:', len(diario))
diario.head()

In [ ]:
pd.plotting.scatter_matrix(diario, figsize=(13, 13), alpha=0.3, color=AZUL,
                           diagonal='hist', hist_kwds={'color': AZUL, 'bins': 30})
plt.suptitle('Matriz de dispersión con datos diarios', y=0.92)
plt.show()

<div style="background-color:#e8f6ef; border-left:6px solid #1baf7a; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**🔎 ¿Qué vemos?**

- **Energía diaria y radiación media**: relación lineal muy estrecha.
- **Energía diaria y producción máxima**: muy relacionadas (casi **redundantes**): un día con un pico alto suele ser un día de mucha energía.
- **Humedad media**: relación negativa con la energía (días húmedos ≈ días nublados).
- **Temperatura media**: nube de puntos sin forma clara: relación débil.
- **Lluvia total**: casi todos los días tienen 0 mm; los días de lluvia suelen tener poca energía.
- En la diagonal, la energía diaria tiene una cola hacia la izquierda (días nublados) y la producción máxima se concentra en torno a 6–7 kWh.

</div>

## 5.4 Mapa de calor: hora del día × día del año

<div style="background-color:#eaf2fc; border-left:6px solid #2a78d6; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**Patrones diarios y anuales en una sola imagen**

Organizamos los datos de un año completo (2016) en una tabla con **una fila por hora del día** y **una columna por día del año**,
usando `pivot_table`. Después la pintamos con `plt.imshow`: cada celda es una hora concreta y su color indica la producción.

Ponemos el fondo en gris: las celdas sin dato (`NaN`) quedan transparentes y se verán **grises**.
Con `interpolation='nearest'` cada celda se pinta como un cuadrado nítido, sin suavizar.

</div>

In [ ]:
datos_2016 = df[df['anyo'] == 2016]
tabla = datos_2016.pivot_table(index='hora', columns='dia_del_anyo', values='produccion')

plt.figure(figsize=(14, 5))
plt.imshow(tabla, aspect='auto', origin='lower', cmap='YlOrRd', interpolation='nearest')
plt.colorbar(label='Producción (kWh)')
plt.gca().set_facecolor('lightgray')
plt.grid(False)

# Marcamos el día 1 de cada mes en el eje x (la columna 0 es el 1 de enero)
inicio_meses = [0, 31, 60, 91, 121, 152, 182, 213, 244, 274, 305, 335]
plt.xticks(inicio_meses, ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic'])
plt.yticks(range(0, 24, 3))
plt.title('Producción FV en 2016: hora del día × día del año')
plt.ylabel('Hora del día')
plt.show()

<div style="background-color:#e8f6ef; border-left:6px solid #1baf7a; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**🔎 ¿Qué vemos?**

- La franja de producción es **más ancha en verano** (≈ 7 a 20 h) y **más estrecha en invierno** (≈ 8 a 17 h): duración del día.
- Las **columnas claras** sueltas son días nublados (como los de la gráfica del apartado 3.4).
- A mediados de **marzo** y a principios de **noviembre** la franja da un **salto de 1 hora**: es el **cambio de hora** (en hora local, el sol «sale» una hora más tarde en verano).
- Los rojos más intensos aparecen entre febrero y abril, **no en verano**: sol + temperaturas suaves = mejor rendimiento.
- La pequeña celda gris de marzo, a las 2 h, es la hora inexistente del cambio de hora.

</div>

## 5.5 Diagramas de caja agrupados

<div style="background-color:#eaf2fc; border-left:6px solid #2a78d6; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**Producción según la hora del día y según el mes**

Con `df.boxplot(column=..., by=...)` pandas dibuja **un diagrama de caja por cada grupo**. Así vemos cómo cambian a la vez
el **nivel** (la mediana) y la **dispersión** (la altura de la caja) entre grupos.

> pandas añade un título automático («Boxplot grouped by...»); lo quitamos con `plt.suptitle('')`.

</div>

In [ ]:
df.boxplot(column='produccion', by='hora', figsize=(14, 4.5), grid=False)
plt.suptitle('')
plt.title('Producción FV por hora del día')
plt.xlabel('Hora del día')
plt.ylabel('Producción (kWh)')
plt.show()

<div style="background-color:#e8f6ef; border-left:6px solid #1baf7a; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**🔎 ¿Qué vemos?**

- Es la misma información del apartado 3.3, pero con cajas: entre las 21 h y las 5 h la producción es prácticamente 0.
- La dispersión (altura de la caja) es máxima entre las 12 h y las 16 h.
- En las horas de transición (7–9 h y 19–20 h) aparecen muchos atípicos por arriba: son días de verano en los que amanece antes o anochece después.
  Aquí «atípico» significa **raro para esa hora**, no erróneo.

</div>

In [ ]:
de_dia.boxplot(column='produccion', by='mes', figsize=(14, 4.5), grid=False)
plt.suptitle('')
plt.title('Producción FV en horas de día, por mes')
plt.xlabel('Mes')
plt.ylabel('Producción (kWh)')
plt.show()

<div style="background-color:#e8f6ef; border-left:6px solid #1baf7a; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**🔎 ¿Qué vemos?**

- Las medianas van de ≈ 1,3 kWh (diciembre) a ≈ 3,2 kWh (julio).
- Pero los **máximos** (bigotes superiores) son más altos de febrero a abril (≈ 9,5 kWh) y más bajos de julio a septiembre (≈ 7 kWh).
- Conclusión: el verano da más energía porque tiene **más horas de sol**, pero con **menos potencia máxima** por el calor.

</div>

<div style="background-color:#eaf2fc; border-left:6px solid #2a78d6; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**Comparar perfiles diarios entre grupos**

Por último, calculamos el **perfil diario mediano** (la mediana de cada hora) para distintos grupos con
`groupby(['hora', grupo])` y `unstack()`, que coloca cada grupo en una columna. Comparamos:

- las **cuatro estaciones**;
- los días **laborables** frente a los **fines de semana**.

> 👀 Este segundo gráfico os será muy útil con los datos de **consumo** de vuestro hogar.

</div>

In [ ]:
perfil_estacion = df.groupby(['hora', 'estacion'])['produccion'].median().unstack()
perfil_finde = df.groupby(['hora', 'fin_de_semana'])['produccion'].median().unstack()

fig, ejes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)

for estacion, color in zip(orden_estaciones, colores_estaciones):
    ejes[0].plot(perfil_estacion.index, perfil_estacion[estacion], color=color, linewidth=2, label=estacion)
ejes[0].set_title('Perfil diario mediano por estación')

ejes[1].plot(perfil_finde.index, perfil_finde[False], color=AZUL, linewidth=2, label='Laborable')
ejes[1].plot(perfil_finde.index, perfil_finde[True], color=NARANJA, linewidth=2, linestyle='--', label='Fin de semana')
ejes[1].set_title('Perfil diario mediano: laborable frente a fin de semana')

ejes[0].set_ylabel('Producción (kWh)')
for eje in ejes:
    eje.set_xlabel('Hora del día')
    eje.set_xticks(range(0, 24, 3))
    eje.legend()

plt.tight_layout()
plt.show()

<div style="background-color:#e8f6ef; border-left:6px solid #1baf7a; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**🔎 ¿Qué vemos?**

- **Por estación**: el verano tiene la curva más alta y ancha (de 6 a 20 h); el invierno, la más estrecha (termina hacia las 17–18 h).
  Todas tienen el máximo hacia las 13–14 h.
- **Laborable frente a fin de semana**: las curvas son prácticamente iguales (las pequeñas diferencias son fruto del azar: hay menos días de fin de semana).
  Es lógico: **al sol no le importa el calendario**.

👉 Con el **consumo** de vuestro hogar, ¡seguramente no será así!

</div>

---
# 6. Conclusiones

<div style="background-color:#e8f6ef; border-left:6px solid #1baf7a; padding:10px 16px; border-radius:6px; color:#1a1a1a;">

**📝 ¿Qué hemos aprendido de estos datos?**

1. **Calidad de los datos**: algo más de 3 años de datos horarios, con **dos huecos de unos dos meses**, horas sueltas que faltan por el **cambio de hora**,
   muy pocos valores vacíos y un **error en el sensor de radiación** (horas con radiación 0 y producción normal).
2. **Dos regímenes, noche y día**: mezclarlos distorsiona la media (1,49 kWh) y la mediana (0,01 kWh). Hay que analizarlos por separado.
3. **De día**, la producción es **asimétrica a la derecha** y muy **variable**: depende de la hora y de las nubes.
4. **Ciclos**: un ciclo **diario** muy marcado (campana con máximo a las 13–14 h) y uno **anual** (máximo en junio–julio y mínimo en noviembre–diciembre).
   **No hay ciclo semanal**.
5. **Temperatura**: con calor los paneles rinden menos; los picos de potencia se dan en primavera, no en verano.
6. **Relaciones**: la radiación (r = 0,94) y la producción pasada (r = 0,92 y 0,86) son las mejores pistas para predecir la producción;
   la humedad tiene una relación negativa; la presión, la lluvia horaria y el día de la semana apenas aportan.
   La relación con la hora del día es muy fuerte pero **no lineal**, y la correlación no la detecta.
7. **Atípicos**: las temperaturas atípicas son episodios de frío reales y se conservan. Los métodos IQR y Z son **criterios**, no verdades absolutas.

</div>

## 🧾 La «receta» del EDA (y el código para cada paso)

Esta tabla resume los pasos que hemos seguido. Son **los mismos** que aplicaréis a vuestros datos de consumo.

| Paso | Pregunta | Código principal |
|---|---|---|
| 1. Cargar | ¿Se leen bien los datos? | `pd.read_csv(..., sep=';')` |
| 2. Conocer | ¿Cuántos datos? ¿Qué significan? | `df.shape`, `df.head()`, `df.rename(...)` |
| 3. Fechas | ¿El tiempo está bien? | `pd.to_datetime(...)`, `df.set_index(...)` |
| 4. Calidad | ¿Faltan valores u horas? | `df.isna().sum()`, `pd.date_range(...)`, `df.asfreq('h')` |
| 5. Nuevas variables | ¿Qué información esconde la fecha? | `df.index.hour`, `.month`, `.dayofweek`, `.map(...)` |
| 6. Tendencia central | ¿Valor típico? | `.mean()`, `.median()`, `.round(0).mode()` |
| 7. Dispersión | ¿Cuánto varía? | `.max() - .min()`, `.var()`, `.std()`, `.quantile(...)` |
| 8. Forma | ¿Simétrica? ¿Colas pesadas? | `.skew()`, `.kurt()` |
| 9. Univariantes | ¿Cómo se distribuye? | `plt.hist`, `plot(kind='kde')`, `plt.boxplot`, `plt.plot` |
| 10. Atípicos | ¿Hay valores raros? ¿Son errores? | método IQR, puntuación Z |
| 11. Tiempo | ¿Tendencias y ciclos? | `resample('D').sum(min_count=...)`, `rolling(...)` |
| 12. Relaciones | ¿Qué variables van juntas? | `plt.scatter`, `.corr()`, `pd.plotting.scatter_matrix` |
| 13. Patrones | ¿Cómo cambia con la hora, el día, la estación? | `pivot_table` + `plt.imshow`, `df.boxplot(by=...)`, `groupby(...).median().unstack()` |

## 📚 Para saber más (opcional)

Existen librerías que generan un informe EDA automático, como **ydata-profiling**. Son útiles para una primera vista rápida,
pero **no sustituyen** a pensar qué preguntas hacer a los datos ni a interpretar los resultados.

```python
# pip install ydata-profiling
from ydata_profiling import ProfileReport
informe = ProfileReport(df[variables], title='Informe EDA - Producción FV')
informe.to_file('informe_eda_pv.html')
```